# Задание 2

In [ ]:
import os
import warnings

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from botocore.client import Config
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_URL = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'oiluser')}:"
    f"{os.getenv('POSTGRES_PASSWORD', 'oilpass')}@"
    f"{PG_HOST}:{os.getenv('POSTGRES_PORT', '5432')}/"
    f"{os.getenv('POSTGRES_DB', 'oildb')}"
)
engine = create_engine(PG_URL)

def minio_endpoint_url():
    raw = os.getenv("MINIO_ENDPOINT", "localhost:9000").strip().rstrip("/")
    if raw.startswith("http://") or raw.startswith("https://"):
        return raw
    return f"http://{raw}"

def minio_client():
    return boto3.client(
        "s3",
        endpoint_url=minio_endpoint_url(),
        aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "minioadmin"),
        aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "minioadmin"),
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split



In [ ]:

dataset = pd.read_sql("""
    SELECT t.well_id, t.date, t.daily_oil_ton,
           COALESCE(AVG(wt.pressure_out), MAX(p.pressure)) AS avg_pressure,
           COALESCE(AVG(wt.temperature), MAX(p.temperature)) AS avg_temperature,
           COALESCE(AVG(wt.pump_current * wt.pressure_out / 1000.0), MAX(p.energy_kwh)/24.0) AS power_kw,
           COALESCE(COUNT(wt.record_id), 24 - COALESCE(MAX(p.downtime_hours),0))::int AS pump_hours
    FROM well_targets t
    LEFT JOIN well_telemetry wt ON wt.well_id = t.well_id AND DATE(wt.timestamp) = t.date
    LEFT JOIN production p ON p.well_id = t.well_id AND p.date = t.date
    GROUP BY t.well_id, t.date, t.daily_oil_ton
""", engine)

features = ["avg_pressure", "avg_temperature", "power_kw", "pump_hours"]
X = dataset[features]
y = dataset["daily_oil_ton"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

lr = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

for name, model in [("LinearRegression", lr), ("RandomForest", rf)]:
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = (mean_squared_error(y_test, pred)) ** 0.5
    print(f"{name}: MAE={mae:.3f}, RMSE={rmse:.3f}, R2={model.score(X_test, y_test):.3f}")

model = rf if rf.score(X_test, y_test) >= lr.score(X_test, y_test) else lr
dataset["predicted"] = model.predict(X)
dataset["error"] = dataset["daily_oil_ton"] - dataset["predicted"]



In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(dataset["daily_oil_ton"], dataset["predicted"], alpha=0.7)
mx = max(dataset["daily_oil_ton"].max(), dataset["predicted"].max())
axes[0].plot([0, mx], [0, mx], "r--")
axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted")
axes[0].set_title("Actual vs Predicted")

dataset.groupby("date")["error"].apply(lambda s: np.abs(s).mean()).plot(ax=axes[1], marker="o")
axes[1].set_title("MAE по времени")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

